# Лабораторная работа 4

**Тема:** аппроксимация табличной функции алгебраическим полиномом методом наименьших квадратов (МНК).

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

Ноутбук запускается без ввода с клавиатуры: по умолчанию (Enter на каждый вопрос) берутся
тестовые точки параболы `y = x^2 + 2x + 1`, поэтому для степени 2 коэффициенты должны
восстановиться почти точно — это и есть проверка корректности программы.


## Импорт библиотек


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## Ввод точек и степени полинома

Степень аппроксимирующего полинома по заданию обязательно вводится с терминала во время
выполнения программы.


In [ ]:
print('Метод наименьших квадратов')
print('Если нажимать Enter, будут взяты тестовые точки параболы y = x^2 + 2x + 1.')

default_points = [(-2, 1), (-1, 0), (0, 1), (1, 4), (2, 9), (3, 16)]

n_raw = input(f'Введите количество точек [{len(default_points)}]: ').strip()
if n_raw:
    n = int(n_raw)
    x_values = []
    y_values = []
    print('Введите точки:')
    for i in range(n):
        x_values.append(float(input(f'x[{i}] = ')))
        y_values.append(float(input(f'y[{i}] = ')))
else:
    x_values = [p[0] for p in default_points]
    y_values = [p[1] for p in default_points]

x_values = np.array(x_values, dtype=float)
y_values = np.array(y_values, dtype=float)

degree = int(input('Введите степень аппроксимирующего полинома [2]: ').strip() or '2')

print('\nТочки:')
for x, y in zip(x_values, y_values):
    print(f'({x:.1f}; {y:.1f})')
print(f'Степень полинома: {degree}')


## Функции

- **`horner`** — вычисление значения полинома по схеме Горнера: `P = (...((c_n*x + c_{n-1})*x + ...)*x + c_0)`,
  что минимизирует число арифметических операций (без явного возведения в степень).
- **`least_squares_polynomial`** — строит систему нормальных уравнений МНК
  (матрица размера `(degree+1) × (degree+1)` из сумм `Σx^(i+j)`, правая часть — из сумм `Σy·x^i`).
- **`gauss_solve`** — решает эту систему методом исключения Гаусса, как рекомендует теория
  (с выбором главного элемента по столбцу для устойчивости). При степени выше 5–7 выводится
  предупреждение: система становится плохо обусловленной.

In [ ]:
def horner(coefficients, x):
    result = coefficients[-1]
    for i in range(len(coefficients) - 2, -1, -1):
        result = result * x + coefficients[i]
    return result


def gauss_solve(A, B):
    # Метод исключения Гаусса с выбором главного элемента по столбцу
    A = A.astype(float).copy()
    B = B.astype(float).copy()
    n = len(B)
    for k in range(n):
        p = k + np.argmax(np.abs(A[k:, k]))
        if abs(A[p, k]) < 1e-14:
            raise ValueError("Система вырождена")
        A[[k, p]], B[[k, p]] = A[[p, k]], B[[p, k]]
        for i in range(k + 1, n):
            m = A[i, k] / A[k, k]
            A[i, k:] -= m * A[k, k:]
            B[i] -= m * B[k]
    C = np.zeros(n)
    for i in range(n - 1, -1, -1):
        C[i] = (B[i] - A[i, i + 1:] @ C[i + 1:]) / A[i, i]
    return C


def least_squares_polynomial(x_values, y_values, degree):
    n = len(x_values)
    if degree >= n:
        raise ValueError("Степень полинома должна быть меньше количества точек")
    if degree > 7:
        print("Внимание: при степени больше 5–7 система плохо обусловлена")

    A = np.zeros((degree + 1, degree + 1))
    B = np.zeros(degree + 1)

    for i in range(degree + 1):
        for j in range(degree + 1):
            A[i, j] = np.sum(x_values ** (i + j))
        B[i] = np.sum(y_values * (x_values ** i))

    return gauss_solve(A, B)

## Коэффициенты полинома


In [ ]:
coefficients = least_squares_polynomial(x_values, y_values, degree)
print("Коэффициенты полинома:")
for i, c in enumerate(coefficients):
    print(f"C{i} = {c:.10f}")

terms = [f"({c:.5f})*x^{i}" if i else f"{c:.5f}" for i, c in enumerate(coefficients)]
print("P(x) = " + " + ".join(terms))


## Проверка качества аппроксимации

Значения полинома в узловых точках вычисляются по схеме Горнера и сравниваются с исходными
экспериментальными значениями — по величине суммарной ошибки судим о качестве аппроксимации.


In [ ]:
total_error = 0
print("x\ty exp\ty МНК\tошибка")
for x, y_real in zip(x_values, y_values):
    y_approx = horner(coefficients, x)
    error = abs(y_real - y_approx)
    total_error += error
    print(f"{x:.1f}\t{y_real:.5f}\t{y_approx:.5f}\t{error:.2e}")
print(f"Суммарная ошибка = {total_error:.10e}")


## График


In [ ]:
x_plot = np.linspace(min(x_values), max(x_values), 500)
y_plot = [horner(coefficients, x) for x in x_plot]

plt.figure(figsize=(10, 6))
plt.scatter(x_values, y_values, label="Экспериментальные точки")
plt.plot(x_plot, y_plot, label=f"МНК полином степени {degree}")
plt.title("Аппроксимация методом наименьших квадратов")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True)
plt.legend()
plt.show()


## Вывод

Метод наименьших квадратов строит полином, минимизирующий сумму квадратов отклонений от
экспериментальных точек. Для тестовой параболы второй степени коэффициенты восстанавливаются
с машинной точностью (суммарная ошибка порядка `1e-14`), что подтверждает корректность
построения и решения системы нормальных уравнений.
